# AIFM Private Debt Fund

This notebook presents a private debt risk-monitoring workflow for a simulated closed-ended AIF portfolio. The fund invests in senior secured loans, high-yield bonds, and CLO tranches.

The analysis focuses on credit and portfolio risk indicators: credit quality, seniority, sector, country, and borrower concentration, the maturity ladder, leverage, credit and rate stress, borrower default stress, investor concentration, and selected sustainability indicators. Because the fund is closed-ended, there is no redemption or asset-liquidity monitoring in this notebook.

> **Output gallery:** All tables and plots generated by this notebook are saved in the [fig/AIFM_PrivateDebt](../../fig/AIFM_PrivateDebt) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

In [ ]:
import warnings

from fund_risk_workflow.data.setup_db import run as setup_db
from fund_risk_workflow.data.mock_bloomberg import MockBloomberg as Bloomberg

import fund_risk_workflow.data.database as db
import fund_risk_workflow.risk.esg_utils as esg_u
import fund_risk_workflow.ui.print_html_utils as phtml
import fund_risk_workflow.ui.private_debt_display as pdd

warnings.filterwarnings("ignore")

setup_db()
ENGINE = db.get_engine()
BBG = Bloomberg()

## 1. Fund Setup and Risk Policy

### 1.1 Fund Example

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'AIFM_PrivateDebt'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---

### 1.2 Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration and used throughout the notebook for measurement, monitoring, and limit checks. The credit stress magnitudes and the senior-secured recovery assumption are documented in the risk policy rather than in notebook code.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

### 1.3 Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# Fixed valuation date for all computations in this notebook
from fund_risk_workflow.config import VALUATION_DATE
VALUATION_DATE

The workflow builder loads positions from the SQLite data layer, enriches them through the simulated Bloomberg workflow, and computes every result used in this notebook. From this point onward, code cells contain only display calls; data aggregation, calculations, and reporting logic remain inside the package modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# Build the full private debt monitoring result set
from fund_risk_workflow.pipeline.private_debt_workflow import build_private_debt_workflow

workflow = build_private_debt_workflow(
    engine=ENGINE,
    bbg=BBG,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
)
NAV = workflow["nav"]
rmp = workflow["rmp"]

---

## 2. Portfolio and Credit Profile

### 2.1 Portfolio Overview

The fund summary, largest positions, and asset-class breakdown describe the invested portfolio at the valuation date.

In [ ]:
phtml.display_fund_summary(FUND_ID, VALUATION_DATE, workflow["positions"], workflow["risk_df"], NAV, export_id="03")

In [ ]:
phtml.display_top_positions(workflow["risk_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(workflow["risk_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")

### 2.2 Credit Quality and Concentration

Credit quality is monitored by rating and seniority, and portfolio concentration by sector, country, and borrower. Borrower exposure uses the full instrument name as the exposure label — the mock data has no separate borrower master.

In [ ]:
pdd.display_rating_profile(workflow["credit_profile"]["rating"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="06")

In [ ]:
pdd.display_seniority_profile(workflow["credit_profile"]["seniority"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="07")

In [ ]:
pdd.display_sector_profile(workflow["concentration"]["sector"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="08")

In [ ]:
pdd.display_country_profile(workflow["concentration"]["country"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="09")

In [ ]:
single_borrower_limit = rmp["concentration_limits_internal"]["single_borrower_max_pct"]
pdd.display_borrower_concentration(
    workflow["concentration"]["borrower"],
    limit_pct=single_borrower_limit,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="10",
)

### 2.3 Maturity Profile

The maturity ladder groups positions by stated maturity. Cash and money-market holdings have no stated maturity and are shown separately. For a closed-ended credit fund, the ladder indicates when principal is expected to return rather than how quickly assets could be sold.

In [ ]:
pdd.display_maturity_ladder(workflow["maturity_profile"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="11")

---

## 3. Leverage

Leverage is monitored using both the Gross and Commitment methods, with the same canonical computation used across the AIF workflows.

- **Gross method**: absolute exposures divided by NAV, with no netting.
- **Commitment method**: recognises eligible hedging and netting arrangements.

The fund holds no derivatives, so both methods reflect the invested credit portfolio; cash is excluded from gross exposure.

In [ ]:
leverage = workflow["leverage"]
phtml.display_leverage(
    leverage["risk_df"],
    leverage["deriv_notional_commitment"],
    leverage["commitment_exposure"],
)

In [ ]:
phtml.display_granular(workflow["granular_leverage"], NAV, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="12")

---

## 4. Credit and Rate Stress

### 4.1 Scenario Stress

Stress testing applies rate, credit spread, combined, and historical scenarios to the current portfolio snapshot using first-order sensitivities:

$$\Delta P_i = \text{sensitivity}_i \times \text{shock}_i \times MV_i$$

The rate and credit shock magnitudes are documented assumptions from the fund's risk policy (migrated from earlier notebook assumptions, unchanged).

In [ ]:
pdd.display_stress_assumptions(workflow["stress_assumptions"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="13")

In [ ]:
pdd.display_stress_results(workflow["stress_results"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="14")

### 4.2 Borrower Default Stress

Borrower default is a primary private-debt risk. The stress assumes the default of each borrower exposure and applies the documented senior-secured recovery assumption from the risk policy to estimate the loss. The worst case is the default of the largest borrower exposure, compared against the single-borrower concentration limit.

In [ ]:
pdd.display_borrower_default(workflow["borrower_default"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="15")

---

## 5. Investor Concentration

Investor concentration is monitored as a closed-ended ownership and governance indicator:

- A single investor above 20% of NAV is flagged per ESMA guidance.
- Top 3 investors above 50% of NAV are flagged as high concentration.

The register is loaded from the fund-level reference data and is simulated. Because the fund is closed-ended with no periodic redemption, concentration is not translated into a redemption scenario.

In [ ]:
pdd.display_investor_concentration_closed_ended(workflow["investor_concentration"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="16")

---

## 6. Sustainability Risk Indicators

Portfolio-level ESG indicators are calculated using NAV-weighted exposures: composite and pillar scores, low-score exposure versus the internal threshold, controversy flags, and carbon intensity.

ESG scores for listed instruments come from the market-data layer. Instruments without a Bloomberg ticker — including the CLO tranches — use reference-data scores; the CLO entries are explicitly simulated manager-estimated look-through scores and are less reliable than third-party assessments.

> Scale note: ESG scores use a 0-100 scale, where 100 is best.

In [ ]:
esg_u.display_esg_assets(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="17")

In [ ]:
esg_u.display_esg_summary(workflow["esg_df"], valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="18")

In [ ]:
esg_u.plot_esg_profile(workflow["esg_df"], FUND_ID, plot_title='ESG profile — Private Debt', valuation_date=VALUATION_DATE, export_id="19")

---

## 7. Annex IV Report

Selected outputs are mapped to Annex IV-style reporting fields. For this closed-ended fund the report covers identification, portfolio breakdown, and leverage detail; liquidity and redemption sections do not apply and are excluded. Investor concentration is monitored separately in Section 5 from the fund-level register.

**Regulatory basis:** Delegated Regulation (EU) 231/2013 Article 110 and Annex IV reporting template.

In [ ]:
import fund_risk_workflow.reporting.annex_iv_workflow as annex_iv_workflow
from fund_risk_workflow.config import QUARTER

annex_iv_result = annex_iv_workflow.run(
    engine=ENGINE,
    fund_id=FUND_ID,
    quarter=QUARTER,
    first_export_id="20",
    sections=("identification", "breakdown", "leverage_detail"),
)